In [7]:
!pip install fasttext

  error: subprocess-exited-with-error
  
  exit code: 1
  
  [119 lines of output]
  C:\Users\giuse\AppData\Local\Temp\pip-build-env-98o6g3wn\overlay\Lib\site-packages\setuptools\dist.py:493: SetuptoolsDeprecationWarning: Invalid dash-separated options
  !!
  
          ********************************************************************************
          Usage of dash-separated 'description-file' will not be supported in future
          versions. Please use the underscore name 'description_file' instead.
  
          This deprecation is overdue, please update your project and remove deprecated
          calls to avoid build errors in the future.
  
          See https://setuptools.pypa.io/en/latest/userguide/declarative_config.html for details.
          ********************************************************************************
  
  !!
    opt = self.warn_dash_deprecation(opt, section)
  running bdist_wheel
  running build
  running build_py
  creating build\lib.win-amd64-c


  Using cached fasttext-0.9.3.tar.gz (73 kB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Using cached pybind11-2.13.6-py3-none-any.whl.metadata (9.5 kB)
Using cached pybind11-2.13.6-py3-none-any.whl (243 kB)
Failed to build fasttext


In [4]:
!pip install DialogTag

  Obtaining dependency information for DialogTag from https://files.pythonhosted.org/packages/30/9b/8ea84ef9bfb6627463dc804ce7a453d2c2663cc9d22844535760ae7d1fac/DialogTag-1.1.3-py3-none-any.whl.metadata
  Using cached DialogTag-1.1.3-py3-none-any.whl.metadata (6.3 kB)
  Obtaining dependency information for transformers>=3.0.0 from https://files.pythonhosted.org/packages/20/37/1f29af63e9c30156a3ed6ebc2754077016577c094f31de7b2631e5d379eb/transformers-4.49.0-py3-none-any.whl.metadata
     ---------------------------------------- 0.0/44.0 kB ? eta -:--:--
     ---------------------------------------- 44.0/44.0 kB 2.3 MB/s eta 0:00:00
  Obtaining dependency information for tensorflow>=2.0.0 from https://files.pythonhosted.org/packages/59/63/5ca1b06cf17dda9c52927917a7911612953a7d91865b1288c7f9eac2b53b/tensorflow-2.18.0-cp310-cp310-win_amd64.whl.metadata
  Obtaining dependency information for tensorflow-intel==2.18.0 from https://files.pythonhosted.org/packages/b2/47/e1a7cc95eccaaa52f47c3de8f


[notice] A new release of pip is available: 23.2.1 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import re
import pickle
import fasttext
import pandas as pd
import numpy as np
import csv
from dialog_tag import DialogTag
from nltk import tokenize, Tree

ModuleNotFoundError: No module named 'fasttext'

In [4]:
from google.colab import drive
drive.mount('/content/drive/')
%cd "MyDrive/RECOVER"

ModuleNotFoundError: No module named 'google.colab'

In [ ]:
import nltk
nltk.download('punkt')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

Specific for our conversation, there is no need if it is just a plain list of sentences


In [ ]:
def preprocessing(path):
    # Read the file
    with open(path, "r+") as conversation:
        conversation = conversation.readlines()
        ## Initial pre-processing
        # Remove empty entries
        conversation = [entry for entry in conversation if entry != '\n']

        # Create speakerturns
        pattern = r"(?P<end_time>\[\d+\:\d+\:\d+\]) (?P<speaker>\w+)\: (?P<content>.+)"
        speakerturns = [re.match(pattern, entry) for entry in conversation]

        # Remove all empty speakerturns
        speakerturns = [turn for turn in speakerturns if turn != None]

        # Change format so each speakerturn has a unique identifier
        for i in range(0, len(speakerturns)):
            current = speakerturns[i]
            speakerturns[i] = (i, current.group('end_time'), current.group('speaker'), current.group('content'))

        df = pd.DataFrame(speakerturns,
                      columns=['identifier', 'time', 'speaker', 'text']
                     ).set_index('identifier', drop=False)
        return df

conversation = preprocessing("example_conversation.txt")

# Questions Identification

In [ ]:
# Speech Act Classification Model
dialog_tagger = DialogTag('bert-base-uncased')

# Whenever a dialogue acts contains "-Question" or is an "Or-Clause", we consider it to be a question.
dialog_acts = ["-Question", "Or-Clause"]

def is_question(s):
  q = False
  for sentence in tokenize.sent_tokenize(s):
    tag = dialog_tagger.predict_tag(sentence)
    # If it contains one of our dialogue act criteria, it is a question
    if any(dialog_act in tag for dialog_act in dialog_acts):
      q = True

  return q


bert-base-uncased found in cache. Loading model...


All model checkpoint layers were used when initializing TFBertForSequenceClassification.

All the layers of TFBertForSequenceClassification were initialized from the model checkpoint at /root/.dialog-tag/models/bert-base-uncased.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFBertForSequenceClassification for predictions without further training.


In [ ]:
ft_model = fasttext.load_model("fasttext_model.bin")
with open('model.pkl', 'rb') as f:
    model= pickle.load(f)

Here we take predictions from our model for each of the turns of the conversation. If a turn is predicted as a requirement, we check if it can be considered as a question. If so, we append to the final list of sentences both the question and the consequent answer - assuming that the answer to a question is in the conversation turn afterwards - as a single sentence. Otherwise the single sentence is appended to the final list.

In [ ]:
from typing_extensions import final
processed_text = [ft_model.get_sentence_vector(sentence) for sentence in conversation['text']]

pred = model.predict(processed_text)

output = []
for i in range(len(pred)):
  if len(conversation.loc[i]['text'].split()) >= 7:
    output.append({"text": conversation.loc[i]['text'], "pred":("Req" if pred[i] == 0 else "NonReq")})

print(output)
final_predicted_rqs = []
for i in range(len(output)):
  if(output[i]["pred"] == "Req"):
    if(is_question(output[i]['text'])):
      final_predicted_rqs.append(output[i]["text"] + "? " + output[i+1]["text"])
    else:
      final_predicted_rqs.append(output[i]["text"])

print(final_predicted_rqs)


/usr/local/lib/python3.10/dist-packages/sklearn/base.py:439: UserWarning: X does not have valid feature names, but SVC was fitted with feature names
  warnings.warn(


[{'text': "we're doing I'm *** and this is **** we are colleagues and uh we are here for the IFA system and all the requirements that we'll now be discussing with you so that we can deliver a good system to you. Yeah. Um Uh just just can you give us 10 seconds? We just need to start recording because. Okay. Okay. So yes. Now the video and our conversation is getting recorded. All right. So we understand that IFA's new system wants to have some key features. For example, you need a proper budget, budgetting portal, reporting portal, scheduling portal and a fan portal for your consumers, fans, users. So we'll go one by one, uh and we'll ask you some questions regarding each of these key categories and then um maybe you can add whatever we have missed and we'll just round it up nicely. So uh regarding the budget, uh We want to know the basics first. So uh how is the IFA Financed and what are your main expense categories? You would say", 'pred': 'Req'}, {'text': "Expense category. So the b

In [ ]:
for sentence in final_predicted_rqs:
  print(sentence)
  print("\n")